# W tej części zajmujemy się załadowaniem i przetworzeniem danych do postaci użytecznej #


## 1. Konfiguracja Środowiska i Import Bibliotek ##

W celu przeprowadzenia analizy ilościowej, wykorzystujemy zestaw bibliotek dedykowanych do obliczeń naukowych, statystyki oraz wizualizacji danych finansowych.

### Stos Technologiczny:

*   **Przetwarzanie Danych:**
    *   `pandas`: Zarządzanie strukturami danych typu DataFrame, obsługa szeregów czasowych.
    *   `numpy`: Operacje wektorowe i funkcje matematyczne (np. logarytmowanie).
*   **Wizualizacja:**
    *   `matplotlib` & `seaborn`: Tworzenie technicznych wykresów cen, wolumenu oraz rozkładów statystycznych.
*   **Ekonometria i Statystyka:**
    *   `statsmodels.tsa`: Biblioteka do analizy szeregów czasowych (Testy stacjonarności ADF, analiza autokorelacji ACF/PACF).
    *   `scipy.stats`: Weryfikacja hipotez statystycznych, w tym testy normalności rozkładu (Jarque-Bera).
*   **Integracja Projektu:**
    *   `sys` & `os`: Konfiguracja ścieżek systemowych umożliwiająca importowanie autorskich modułów z katalogu `src/`.

---

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
import os

# Dodanie folderu src do ścieżki (wyjście z notebooks/ do głównego katalogu)
sys.path.append(os.path.abspath('../'))
from src.data_loader import load_data, clean_financial_data, analyze_data_content

# Ustawienia estetyczne wykresów
%matplotlib inline
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = [12, 6]
plt.rcParams['figure.dpi'] = 100

## 2. Wczytywanie i czyszczenie danych ##

Ładowanie danych

In [2]:
from pathlib import Path

# To znajdzie główny folder projektu (BICC_2026), 
# idąc od miejsca, gdzie jest ten notebook o dwa poziomy w górę
BASE_DIR = Path(os.getcwd()).parent
path_daily = BASE_DIR / "data" / "raw" / "BICC_Daily_Markets_Anon.csv"
path_weekly = BASE_DIR / "data" / "raw" / "BICC_Weekly_Macro_Anon.csv"
path_monthly = BASE_DIR / "data" / "raw" / "BICC_Monthly_Macro_Anon.csv"


data_daily = load_data(str(path_daily))
data_weekly = load_data(str(path_weekly))
data_monthly = load_data(str(path_monthly))

✅ Ustawiono indeks czasowy na kolumnie: Date
✅ Ustawiono indeks czasowy na kolumnie: Date
✅ Ustawiono indeks czasowy na kolumnie: Date


Analiza zawartości danych dziennych

In [3]:
print(data_daily.info())

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 1566 entries, 2020-01-01 to 2025-12-31
Data columns (total 20 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   Target_Asset_Close           1566 non-null   float64
 1   Target_Asset_Volume          1565 non-null   float64
 2   Risk_Index_Close             1565 non-null   float64
 3   Risk_Index_Volume            1565 non-null   float64
 4   Safe_Haven_Metal_Close       1565 non-null   float64
 5   Safe_Haven_Metal_Volume      1565 non-null   float64
 6   Commodity_Energy_1_Close     1565 non-null   float64
 7   Commodity_Energy_1_Volume    1565 non-null   float64
 8   Bond_Yield_RegionA_Close     1565 non-null   float64
 9   Bond_Yield_RegionA_Volume    1565 non-null   float64
 10  Equity_Index_RegionB_Close   1564 non-null   float64
 11  Equity_Index_RegionB_Volume  1564 non-null   float64
 12  Equity_Index_RegionA_Close   1565 non-null   float64
 13  

In [4]:

# Czyszczenie: wypełnianie luk w cenach ( Close ) zgodnie z instrukcją
close_cols = [col for col in data_daily.columns if '_Close' in col]

# Czyszczenie: zamiana NA w Volume na 0 (jeśli tak zdecydujecie)
volume_cols = [col for col in data_daily.columns if '_Volume' in col]

print(close_cols)
print(volume_cols)

['Target_Asset_Close', 'Risk_Index_Close', 'Safe_Haven_Metal_Close', 'Commodity_Energy_1_Close', 'Bond_Yield_RegionA_Close', 'Equity_Index_RegionB_Close', 'Equity_Index_RegionA_Close', 'Commodity_Energy_2_Close', 'Supply_Chain_Index_Close', 'ESG_Asset_Close']
['Target_Asset_Volume', 'Risk_Index_Volume', 'Safe_Haven_Metal_Volume', 'Commodity_Energy_1_Volume', 'Bond_Yield_RegionA_Volume', 'Equity_Index_RegionB_Volume', 'Equity_Index_RegionA_Volume', 'Commodity_Energy_2_Volume', 'Supply_Chain_Index_Volume', 'ESG_Asset_Volume']


In [5]:
# braki danych dla kolumn z cenami
data_daily[close_cols].isna().sum()

Target_Asset_Close            0
Risk_Index_Close              1
Safe_Haven_Metal_Close        1
Commodity_Energy_1_Close      1
Bond_Yield_RegionA_Close      1
Equity_Index_RegionB_Close    2
Equity_Index_RegionA_Close    1
Commodity_Energy_2_Close      1
Supply_Chain_Index_Close      1
ESG_Asset_Close               4
dtype: int64

In [6]:
# braki danych dla wolumenow
data_daily[volume_cols].isna().sum()

Target_Asset_Volume               1
Risk_Index_Volume                 1
Safe_Haven_Metal_Volume           1
Commodity_Energy_1_Volume         1
Bond_Yield_RegionA_Volume         1
Equity_Index_RegionB_Volume       2
Equity_Index_RegionA_Volume       1
Commodity_Energy_2_Volume         1
Supply_Chain_Index_Volume      1566
ESG_Asset_Volume                  4
dtype: int64

In [7]:
data_daily.describe()

,Target_Asset_Close,Target_Asset_Volume,Risk_Index_Close,Risk_Index_Volume,Safe_Haven_Metal_Close,Safe_Haven_Metal_Volume,Commodity_Energy_1_Close,Commodity_Energy_1_Volume,Bond_Yield_RegionA_Close,Bond_Yield_RegionA_Volume,Equity_Index_RegionB_Close,Equity_Index_RegionB_Volume,Equity_Index_RegionA_Close,Equity_Index_RegionA_Volume,Commodity_Energy_2_Close,Commodity_Energy_2_Volume,Supply_Chain_Index_Close,Supply_Chain_Index_Volume,ESG_Asset_Close,ESG_Asset_Volume
count,1566.000000,1565.000000,1565.000000,1565.0,1565.000000,1565.000000,1565.000000,1565.000000,1565.000000,1565.0,1564.000000,1.564000e+03,1565.000000,1.565000e+03,1565.000000,1565.000000,1565.000000,0.0,1562.000000,1562.000000
mean,1.112200,78571.145687,20.992728,0.0,2190.297186,4801.700319,73.834083,31714.592332,2.954323,0.0,4265.520355,3.169160e+07,15553.410973,5.602225e+09,50.409842,1407.974441,1787.083067,NaN,64.357478,13178.181818
std,0.057365,126944.464871,7.944161,0.0,640.479643,23816.731389,18.802197,14391.667508,1.407893,0.0,736.962738,1.542611e+07,4365.611891,1.931356e+09,46.800256,3832.742859,825.311758,NaN,21.401190,12221.695364
min,0.959619,0.000000,11.860000,0.0,1477.300049,0.000000,19.330000,0.000000,0.499000,0.0,2385.820068,0.000000e+00,6994.290039,2.169020e+09,3.510000,0.000000,393.000000,NaN,16.120000,0.000000
25%,1.074212,476.000000,15.920000,0.0,1795.900024,95.000000,64.910004,22952.000000,1.530000,0.0,3726.047424,2.264640e+07,12268.320312,4.406880e+09,25.809000,60.000000,1294.000000,NaN,53.792500,1440.000000
50%,1.099741,960.000000,19.000000,0.0,1918.400024,304.000000,75.129997,30426.000000,3.572000,0.0,4197.214844,2.809410e+07,14784.299805,5.028410e+09,35.190000,350.000000,1685.000000,NaN,70.135000,12810.000000
75%,1.164114,162990.000000,24.090000,0.0,2372.199951,853.000000,84.459999,38711.000000,4.206000,0.0,4898.804810,3.530952e+07,18796.019531,6.243760e+09,51.726000,980.000000,2102.000000,NaN,80.752500,22150.000000
max,1.234111,648642.000000,82.690002,0.0,4529.100098,251274.000000,127.980003,127044.000000,4.988000,0.0,5796.220215,1.673299e+08,26119.849609,1.630873e+10,321.415000,47190.000000,5650.000000,NaN,97.470000,78660.000000


In [8]:
# usuniecie danych z pustymi wolumenami (0 lub NA)
data_daily = data_daily.drop(columns=['Risk_Index_Volume', 'Bond_Yield_RegionA_Volume', 'Supply_Chain_Index_Volume'])

In [9]:
# analiza brakow danych

# od kiedy mamy komplet danych dla wszystkich kolumn
first_valid = data_daily.apply(lambda col: col.first_valid_index()).max()
print(f"Dane stają się kompletne od: {first_valid}")

# przycinamy zbior danych
data_daily = data_daily.loc[first_valid:]

Dane stają się kompletne od: 2020-01-07 00:00:00


### Stworzenie ramki danych z samymi zwrotami ###

In [10]:
# 1. Tworzymy nową ramkę na zwroty
df_returns = pd.DataFrame(index=data_daily.index)

# 2. Obliczamy Log Returns / Diffs
for col in data_daily.columns:
    if '_Close' in col:
        if 'Bond_Yield' in col:
            # Dla obligacji zmiana punktowa
            df_returns[f'{col}_diff'] = data_daily[col].diff()
        else:
            # Log Returns dla reszty
            df_returns[f'{col}_log_ret'] = np.log(data_daily[col] / data_daily[col].shift(1))

# 3. Usuwamy pierwszy wiersz, który powstał przez .diff() / .shift(1)
df_returns = df_returns.dropna()

### Analiza ramki danych ze zwrotami ###

In [11]:
df_returns.info()

df_returns.describe()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 1561 entries, 2020-01-08 to 2025-12-31
Data columns (total 10 columns):
 #   Column                              Non-Null Count  Dtype  
---  ------                              --------------  -----  
 0   Target_Asset_Close_log_ret          1561 non-null   float64
 1   Risk_Index_Close_log_ret            1561 non-null   float64
 2   Safe_Haven_Metal_Close_log_ret      1561 non-null   float64
 3   Commodity_Energy_1_Close_log_ret    1561 non-null   float64
 4   Bond_Yield_RegionA_Close_diff       1561 non-null   float64
 5   Equity_Index_RegionB_Close_log_ret  1561 non-null   float64
 6   Equity_Index_RegionA_Close_log_ret  1561 non-null   float64
 7   Commodity_Energy_2_Close_log_ret    1561 non-null   float64
 8   Supply_Chain_Index_Close_log_ret    1561 non-null   float64
 9   ESG_Asset_Close_log_ret             1561 non-null   float64
dtypes: float64(10)
memory usage: 134.1 KB


,Target_Asset_Close_log_ret,Risk_Index_Close_log_ret,Safe_Haven_Metal_Close_log_ret,Commodity_Energy_1_Close_log_ret,Bond_Yield_RegionA_Close_diff,Equity_Index_RegionB_Close_log_ret,Equity_Index_RegionA_Close_log_ret,Commodity_Energy_2_Close_log_ret,Supply_Chain_Index_Close_log_ret,ESG_Asset_Close_log_ret
count,1561.000000,1561.000000,1561.000000,1561.000000,1561.000000,1561.000000,1561.000000,1561.000000,1561.000000,1561.000000
mean,0.000031,0.000052,0.000649,-0.000074,0.001496,0.000277,0.000672,0.000550,0.000554,0.000788
std,0.004718,0.077531,0.010441,0.026296,0.060587,0.012740,0.015896,0.054008,0.036554,0.025428
min,-0.028144,-0.442449,-0.059062,-0.279761,-0.322000,-0.132405,-0.130032,-0.352414,-0.192272,-0.177347
25%,-0.002619,-0.038009,-0.003576,-0.009221,-0.030000,-0.003972,-0.004861,-0.022744,-0.018640,-0.010918
50%,-0.000097,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,0.002846,0.025159,0.005587,0.011580,0.033000,0.005751,0.008140,0.022185,0.016791,0.014219
max,0.027537,0.729851,0.057775,0.190774,0.322000,0.088343,0.113528,0.412788,0.203367,0.161378


## 3. Analiza zbioru danych tygodniowych ##

### Wstępne sprawdzenie ###

In [12]:
data_weekly.head()

,Macro_Labor_RegionA,Macro_Stress_RegionA
Date,,
2020-01-04,224000,-0.2339
2020-01-11,207000,-0.3600
2020-01-18,221000,-0.5282
2020-01-25,210000,-0.4169
2020-02-01,207000,-0.1290


In [13]:
data_weekly.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 313 entries, 2020-01-04 to 2025-12-27
Data columns (total 2 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Macro_Labor_RegionA   313 non-null    int64  
 1   Macro_Stress_RegionA  313 non-null    float64
dtypes: float64(1), int64(1)
memory usage: 7.3 KB


### Usunięcie kolumny z samymi NA i dodatkowe zmienne ###

In [14]:
data_weekly_processed = data_weekly.copy()

# dodatkowa zmienna - zmiana procentowa zatrudnienia
data_weekly_processed['Macro_Labor_RegionA_change'] = data_weekly_processed['Macro_Labor_RegionA'].pct_change()

data_weekly_processed.dropna(inplace = True)
data_weekly_processed.head()

,Macro_Labor_RegionA,Macro_Stress_RegionA,Macro_Labor_RegionA_change
Date,,,
2020-01-11,207000,-0.3600,-0.075893
2020-01-18,221000,-0.5282,0.067633
2020-01-25,210000,-0.4169,-0.049774
2020-02-01,207000,-0.1290,-0.014286
2020-02-08,202000,-0.4676,-0.024155


In [15]:
data_weekly_processed.describe()

,Macro_Labor_RegionA,Macro_Stress_RegionA,Macro_Labor_RegionA_change
count,3.120000e+02,312.000000,312.000000
mean,4.512147e+05,-0.330527,0.025927
std,6.980878e+05,0.731291,0.554292
min,1.900000e+05,-0.975400,-0.206616
25%,2.150000e+05,-0.704500,-0.039601
50%,2.260000e+05,-0.496900,-0.008889
75%,3.367500e+05,-0.227175,0.019280
max,6.137000e+06,5.668300,9.673993


## 4. Złączenie danych dziennych i tygodniowych ##

In [16]:
# 1. Sortujemy indeksy (wymóg dla merge_asof)
df_returns = df_returns.sort_index()
data_weekly_processed = data_weekly_processed.sort_index()

# 2. łączenie
# 'backward' sprawi, że w poniedziałek 13.01 model będzie widział dane z soboty 11.01,
# ale nie dowie się nic o danych z przyszłej soboty 18.01.
df_combined = pd.merge_asof(
    data_daily, 
    data_weekly_processed, 
    left_index=True, 
    right_index=True, 
    direction='backward'
)

# 3. Wypełnienie luk (ffill)
# Dane tygodniowe "wskakują" raz na tydzień, przez resztę dni musimy je powtarzać.
df_combined['Macro_Labor_RegionA_change'] = df_combined['Macro_Labor_RegionA_change'].ffill()

In [17]:
df_combined.head()

,Target_Asset_Close,Target_Asset_Volume,Risk_Index_Close,Safe_Haven_Metal_Close,Safe_Haven_Metal_Volume,Commodity_Energy_1_Close,Commodity_Energy_1_Volume,Bond_Yield_RegionA_Close,Equity_Index_RegionB_Close,Equity_Index_RegionB_Volume,Equity_Index_RegionA_Close,Equity_Index_RegionA_Volume,Commodity_Energy_2_Close,Commodity_Energy_2_Volume,Supply_Chain_Index_Close,ESG_Asset_Close,ESG_Asset_Volume,Macro_Labor_RegionA,Macro_Stress_RegionA,Macro_Labor_RegionA_change
Date,,,,,,,,,,,,,,,,,,,,
2020-01-07,1.119799,1850.0,13.79,1571.800049,47.0,68.269997,41178.0,1.827,3759.250000,29853300.0,8846.450195,2.381740e+09,11.930,520.0,791.0,25.06,20.0,NaN,NaN,NaN
2020-01-08,1.115474,850.0,13.45,1557.400024,236.0,65.440002,85232.0,1.874,3772.560059,34988400.0,8912.370117,2.472620e+09,11.965,640.0,773.0,24.60,510.0,NaN,NaN,NaN
2020-01-09,1.111321,1296.0,12.54,1551.699951,54.0,65.370003,50520.0,1.858,3795.879883,33480100.0,8989.629883,2.540960e+09,12.180,1140.0,772.0,25.20,120.0,NaN,NaN,NaN
2020-01-10,1.111111,1115.0,12.56,1557.500000,16.0,64.980003,46595.0,1.825,3789.520020,27400900.0,8966.639648,2.382900e+09,11.940,1150.0,774.0,24.73,60.0,NaN,NaN,NaN
2020-01-13,1.111667,10583.0,12.32,1548.400024,48.0,64.199997,35430.0,1.848,3779.679932,27508600.0,9070.650391,2.536820e+09,12.055,400.0,765.0,24.66,110.0,207000.0,-0.36,-0.075893


## 5. Analiza zbioru danych miesiecznych ##

In [18]:
monthly_data = data_monthly.copy()
monthly_data.head()

,Macro_Inflation_RegionA,Macro_Inflation_RegionB,Macro_Production_RegionA,Macro_Production_RegionB,Macro_Global_Uncertainty
Date,,,,,
2020-01-01,259.127,104.37,101.0338,99.2,191.721791
2020-01-01,259.127,104.37,101.0338,99.2,191.721791
2020-01-01,259.127,104.37,101.0338,99.2,191.721791
2020-02-01,259.250,104.56,101.3735,98.6,196.493257
2020-02-01,259.250,104.56,101.3735,98.6,196.493257


In [19]:
monthly_data.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 216 entries, 2020-01-01 to 2025-12-01
Data columns (total 5 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Macro_Inflation_RegionA   213 non-null    float64
 1   Macro_Inflation_RegionB   216 non-null    float64
 2   Macro_Production_RegionA  216 non-null    float64
 3   Macro_Production_RegionB  216 non-null    float64
 4   Macro_Global_Uncertainty  213 non-null    float64
dtypes: float64(5)
memory usage: 10.1 KB


### załatwienie NA przez forward fill ###

In [20]:

monthly_data = monthly_data.ffill()

print(monthly_data.info())

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 216 entries, 2020-01-01 to 2025-12-01
Data columns (total 5 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Macro_Inflation_RegionA   216 non-null    float64
 1   Macro_Inflation_RegionB   216 non-null    float64
 2   Macro_Production_RegionA  216 non-null    float64
 3   Macro_Production_RegionB  216 non-null    float64
 4   Macro_Global_Uncertainty  216 non-null    float64
dtypes: float64(5)
memory usage: 10.1 KB
None


### Dodanie nowych kolumn - zmian w czasie ###

In [21]:
monthly_data['Macro_Inflation_RegionA_change'] = monthly_data['Macro_Inflation_RegionA'].pct_change()
monthly_data['Macro_Inflation_RegionB_change'] = monthly_data['Macro_Inflation_RegionB'].pct_change()

monthly_data['Macro_Production_RegionA_change'] = monthly_data['Macro_Production_RegionA'].diff()
monthly_data['Macro_Production_RegionB_change'] = monthly_data['Macro_Production_RegionB'].diff()

monthly_data['Macro_Global_Uncertainty_change'] = monthly_data['Macro_Global_Uncertainty'].diff()

print(monthly_data.head())



            Macro_Inflation_RegionA  Macro_Inflation_RegionB  \
Date                                                           
2020-01-01                  259.127                   104.37   
2020-01-01                  259.127                   104.37   
2020-01-01                  259.127                   104.37   
2020-02-01                  259.250                   104.56   
2020-02-01                  259.250                   104.56   

            Macro_Production_RegionA  Macro_Production_RegionB  \
Date                                                             
2020-01-01                  101.0338                      99.2   
2020-01-01                  101.0338                      99.2   
2020-01-01                  101.0338                      99.2   
2020-02-01                  101.3735                      98.6   
2020-02-01                  101.3735                      98.6   

            Macro_Global_Uncertainty  Macro_Inflation_RegionA_change  \
Date            

In [22]:
nowe_kolumny = ['Macro_Inflation_RegionA_change',
       'Macro_Inflation_RegionB_change', 'Macro_Production_RegionA_change',
       'Macro_Production_RegionB_change', 'Macro_Global_Uncertainty_change']
for col in nowe_kolumny:
    print('*' * 100)
    print(monthly_data[col].value_counts())

****************************************************************************************************
Macro_Inflation_RegionA_change
 0.000000    145
 0.000475      1
-0.004528      1
-0.007920      1
-0.000898      1
            ... 
 0.002284      1
 0.003483      1
 0.002951      1
 0.002523      1
 0.002978      1
Name: count, Length: 71, dtype: int64
****************************************************************************************************
Macro_Inflation_RegionB_change
 0.000000    39
-0.000095     2
 0.000474     2
-0.000395     2
-0.000096     1
             ..
 0.000619     1
 0.000077     1
 0.001160     1
 0.000541     1
 0.000077     1
Name: count, Length: 174, dtype: int64
****************************************************************************************************
Macro_Production_RegionA_change
 0.0000     144
 0.3397       1
-3.9657       1
-12.8459      1
 1.3985       1
           ... 
-0.2693       1
 0.0433       1
-0.4485       1
-0.1385       1
 0.

### Usunięcie pierwszych miesięcy - mamy NA ###

In [23]:
monthly_data.dropna(inplace = True)

## 5. Połączenie danych miesięcznych z połączoną resztą danych ##

In [24]:
# Łączymy bazę (Daily + Weekly) z nowymi danymi Monthly
df_final = pd.merge_asof(
    df_combined.sort_index(), 
    monthly_data.sort_index(), 
    left_index=True, 
    right_index=True, 
    direction='backward' # Model widzi ostatni raport miesięczny
)

df_final.head()

,Target_Asset_Close,Target_Asset_Volume,Risk_Index_Close,Safe_Haven_Metal_Close,Safe_Haven_Metal_Volume,Commodity_Energy_1_Close,Commodity_Energy_1_Volume,Bond_Yield_RegionA_Close,Equity_Index_RegionB_Close,Equity_Index_RegionB_Volume,...,Macro_Inflation_RegionA,Macro_Inflation_RegionB,Macro_Production_RegionA,Macro_Production_RegionB,Macro_Global_Uncertainty,Macro_Inflation_RegionA_change,Macro_Inflation_RegionB_change,Macro_Production_RegionA_change,Macro_Production_RegionB_change,Macro_Global_Uncertainty_change
Date,,,,,,,,,,,,,,,,,,,,,
2020-01-07,1.119799,1850.0,13.79,1571.800049,47.0,68.269997,41178.0,1.827,3759.250000,29853300.0,...,259.127,104.37,101.0338,99.2,191.721791,0.0,0.0,0.0,0.0,0.0
2020-01-08,1.115474,850.0,13.45,1557.400024,236.0,65.440002,85232.0,1.874,3772.560059,34988400.0,...,259.127,104.37,101.0338,99.2,191.721791,0.0,0.0,0.0,0.0,0.0
2020-01-09,1.111321,1296.0,12.54,1551.699951,54.0,65.370003,50520.0,1.858,3795.879883,33480100.0,...,259.127,104.37,101.0338,99.2,191.721791,0.0,0.0,0.0,0.0,0.0
2020-01-10,1.111111,1115.0,12.56,1557.500000,16.0,64.980003,46595.0,1.825,3789.520020,27400900.0,...,259.127,104.37,101.0338,99.2,191.721791,0.0,0.0,0.0,0.0,0.0
2020-01-13,1.111667,10583.0,12.32,1548.400024,48.0,64.199997,35430.0,1.848,3779.679932,27508600.0,...,259.127,104.37,101.0338,99.2,191.721791,0.0,0.0,0.0,0.0,0.0


### Usunięcie NA z finalnego zbioru danych ###

In [25]:
df_final.dropna(inplace = True)
print(df_final.info())

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 1558 entries, 2020-01-13 to 2025-12-31
Data columns (total 30 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   Target_Asset_Close               1558 non-null   float64
 1   Target_Asset_Volume              1558 non-null   float64
 2   Risk_Index_Close                 1558 non-null   float64
 3   Safe_Haven_Metal_Close           1558 non-null   float64
 4   Safe_Haven_Metal_Volume          1558 non-null   float64
 5   Commodity_Energy_1_Close         1558 non-null   float64
 6   Commodity_Energy_1_Volume        1558 non-null   float64
 7   Bond_Yield_RegionA_Close         1558 non-null   float64
 8   Equity_Index_RegionB_Close       1558 non-null   float64
 9   Equity_Index_RegionB_Volume      1558 non-null   float64
 10  Equity_Index_RegionA_Close       1558 non-null   float64
 11  Equity_Index_RegionA_Volume      1558 non-null   float64
 12  Co

### Macierz korelacji dla danych finalnych ###

In [27]:
# 3. Macierz korelacji - co rusza Twoim assetem?
corrs = df_final.corr()['Target_Asset_Close'].sort_values(ascending=False)
print("Korelacja z jutrzejszym zwrotem Targetu:")
print(corrs)

Korelacja z jutrzejszym zwrotem Targetu:
Target_Asset_Close                 1.000000
Supply_Chain_Index_Close           0.333244
Safe_Haven_Metal_Close             0.153645
Macro_Labor_RegionA                0.132712
Equity_Index_RegionA_Volume        0.108344
Commodity_Energy_1_Volume          0.105773
Equity_Index_RegionA_Close         0.092443
Macro_Global_Uncertainty           0.065510
Safe_Haven_Metal_Volume            0.013363
Equity_Index_RegionB_Close         0.000336
Equity_Index_RegionB_Volume       -0.023370
Macro_Labor_RegionA_change        -0.040407
Risk_Index_Close                  -0.041271
Commodity_Energy_2_Volume         -0.052316
Target_Asset_Volume               -0.053628
Macro_Production_RegionB          -0.070618
Macro_Stress_RegionA              -0.141338
Macro_Production_RegionA          -0.180482
Macro_Inflation_RegionB_change    -0.260104
ESG_Asset_Volume                  -0.263336
Macro_Inflation_RegionA           -0.412815
Macro_Inflation_RegionB           -

## 6. Zapisanie plikow csv do folderu data/processed ##

In [28]:


# 1. ścieżka do folderu
output_dir = '../data/processed'
if not os.path.exists(output_dir):
    os.makedirs(output_dir)
    print(f"Utworzono folder: {output_dir}")

# 2. Słownik ramek do zapisania
dfs_to_save = {
    'data_daily_cleaned.csv': data_daily,       # Surowe, ale oczyszczone z NaN i ffill
    'df_returns_daily.csv': df_returns,         # Same zwroty dzienne + TARGET
    'data_weekly_macro.csv': data_weekly,       # Oryginalne dane tygodniowe
    'data_monthly_macro.csv': monthly_data,     # Oryginalne dane miesięczne
    'df_final_master.csv': df_final             # Zintegrowany zbiór (Daily + Weekly + Monthly)
}

# 3. Pętla zapisująca pliki
for filename, df in dfs_to_save.items():
    path = os.path.join(output_dir, filename)
    df.to_csv(path)
    print(f"Zapisano: {path} | Wymiary: {df.shape}")

print("\n--- Wszystkie pliki są gotowe ---")

Zapisano: ../data/processed\data_daily_cleaned.csv | Wymiary: (1562, 17)
Zapisano: ../data/processed\df_returns_daily.csv | Wymiary: (1561, 10)
Zapisano: ../data/processed\data_weekly_macro.csv | Wymiary: (313, 2)
Zapisano: ../data/processed\data_monthly_macro.csv | Wymiary: (215, 10)
Zapisano: ../data/processed\df_final_master.csv | Wymiary: (1558, 30)

--- Wszystkie pliki są gotowe ---
